# Phase 1: zero-shot Whisper baseline on Cypriot Greek

Run on a Colab T4 (Runtime > Change runtime type > T4 GPU). Before starting, run `bash scripts/make_colab_bundle.sh` locally and upload `cyasr_bundle.zip` to Google Drive (`MyDrive/cyasr/`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BUNDLE = '/content/drive/MyDrive/cyasr/cyasr_bundle.zip'
DRIVE_RUNS = '/content/drive/MyDrive/cyasr/runs'

!rm -rf /content/cyasr && mkdir -p /content/cyasr
!unzip -q {BUNDLE} -d /content/cyasr
%cd /content/cyasr
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# torch is already installed on Colab, so only the other requirements are needed
!pip install -q -r requirements.txt

In [ ]:
# Smoke test first (5 clips); expect a minute or two
!python -m cyasr.run_baseline --config configs/baseline_whisper-tiny.yaml

In [ ]:
# Full runs on dev + test, greedy and with temperature fallback.
# Order: small -> medium -> large-v3. Results are copied to Drive after each model,
# so a disconnect doesn't lose earlier runs.
for m in ['small', 'medium', 'large-v3']:
    for dec in ['greedy', 'fallback']:
        !python -m cyasr.run_baseline --config configs/baseline_whisper-{m}.yaml --decoding {dec}
    !mkdir -p {DRIVE_RUNS} && cp -r runs/* {DRIVE_RUNS}/

In [ ]:
!python scripts/summarize_runs.py
!python -m cyasr.analyze runs/baseline_whisper-small_fallback_dev --top 25

In [ ]:
# Copy results back to Drive (Colab disk is wiped when the runtime ends)
!mkdir -p {DRIVE_RUNS} && cp -r runs/* {DRIVE_RUNS}/